## Step 5 : Do the rankings hold?

Reads the 3 ball tables from steps 1 to 3 and the two ranked lists from step 4 (`outputs/04_batters.csv`, `outputs/04_bowlers.csv`).
Writes `outputs/05_batters.csv` and `outputs/05_bowlers.csv` (the final lists, ranked in IPL terms, with how sure I am of every rank),
`outputs/05_splithalf.csv`, `outputs/05_blend.csv` and `outputs/05_sensitivity.csv` (the tables behind the report).

Step 4 ranked both lists on level. This step tries to break that ranking in 5 ways:
1. Split-half : does one half of the matches predict the other half? For level, for clutch, and for a blend of the two
2. IPL against T20I : do the two competitions agree about a player, and is a T20I ball easier?
3. Rule index : does the list change if pressure is the simple rule from step 2?
4. Eligibility : does the list change with other minimums?
5. My own choices : the price of a wicket, the years used, and the shrinking

Then the final lists : the level in IPL terms (the T20I gap from test 2 taken off), and how often each player is in the top 10 when the matches are drawn again 1,000 times.

In [28]:
# Importing Libraries
import os
import numpy as np
import pandas as pd

output_folder = os.path.join("..", "outputs")
keys = ['match_id', 'innings', 'innings_ball_num']

# Loading Dataset (the 3 tables from steps 1 to 3, one row per ball)
balls = pd.read_parquet(os.path.join(output_folder, "balls_clean.parquet"), columns=keys + [
    'striker_id', 'striker_name', 'bowler_id', 'bowler_name', 'legal', 'comp_group'])
balls = balls.merge(pd.read_parquet(os.path.join(output_folder, "balls_pressure.parquet"), columns=keys + ['pressure', 'rule_index']), on=keys)
balls = balls.merge(pd.read_parquet(os.path.join(output_folder, "balls_value.parquet"), columns=keys + ['batter_value', 'bowler_value']), on=keys)

# Same as step 4 : Other T20 players are not ranked, a high pressure ball has pressure 80 or more
balls = balls[balls['comp_group'] != 'Other T20']
balls['high'] = balls['pressure'] >= 80

# For test 4 : the top fifth of balls again, but picked with the simple rule index from step 2
balls['high_rule'] = balls['rule_index'] >= balls['rule_index'].quantile(0.8)
print(balls.shape)
print("share of high pressure balls:", round(balls['high'].mean(), 3), "with the rule index:", round(balls['high_rule'].mean(), 3),
      "balls that are high in both:", round((balls['high'] & balls['high_rule']).mean(), 3))

# The two ranked lists from step 4 (index = player id), and a name for every player id
step4 = {'batters': pd.read_csv(os.path.join(output_folder, "04_batters.csv"), index_col=0),
         'bowlers': pd.read_csv(os.path.join(output_folder, "04_bowlers.csv"), index_col=0)}
names = {'batters': balls.groupby('striker_id')['striker_name'].first(),
         'bowlers': balls.groupby('bowler_id')['bowler_name'].first()}

(193988, 15)
share of high pressure balls: 0.198 with the rule index: 0.2 balls that are high in both: 0.071


In [29]:
# One row per player per match, with every sum the tests need.
# Every test below is a different way of adding these rows up (one half of the matches, IPL only,
# matches drawn again with repeats), so no test has to go back to the 194,000 balls.
def match_rows(source, player, value):
    source = source.assign(square=source[value] ** 2, high_value=source[value] * source['high'],
                           rule_value=source[value] * source['high_rule'])
    rows = source.groupby([player, 'match_id']).agg(
        balls=(value, 'size'), total=(value, 'sum'), squares=('square', 'sum'),
        high_balls=('high', 'sum'), high_total=('high_value', 'sum'),
        rule_balls=('high_rule', 'sum'), rule_total=('rule_value', 'sum'),
        comp_group=('comp_group', 'first')).reset_index()
    ids = np.sort(rows[player].unique())
    rows['code'] = np.searchsorted(ids, rows[player])
    rows['match_code'] = rows['match_id'].astype('category').cat.codes
    return rows, ids


# Adds the rows up to one line per player. weight = how many times each row counts (once if not given).
# The lines are in the order of the player codes, so line 0 is the player with code 0.
def player_numbers(rows, players, weight=None):
    weight = np.ones(len(rows)) if weight is None else weight
    code = rows['code'].values
    table = pd.DataFrame({column: np.bincount(code, weight * rows[column].values, minlength=players) for column in [
        'balls', 'total', 'squares', 'high_balls', 'high_total', 'rule_balls', 'rule_total']})
    other_balls = table['balls'] - table['high_balls']
    table['raw_level'] = table['total'] / table['balls']
    table['raw_high'] = table['high_total'] / table['high_balls']
    table['delta'] = table['raw_high'] - (table['total'] - table['high_total']) / other_balls

    # Pressure match = 6 or more high pressure balls in the match. beat_average = share of them above the average player
    pressure_match = rows['high_balls'].values >= 6
    table['pressure_matches'] = np.bincount(code, weight * pressure_match, minlength=players)
    table['beat_average'] = np.bincount(code, weight * (pressure_match & (rows['high_total'].values > 0)), minlength=players) / table['pressure_matches']
    return table


# The shrunk level of step 4, from the match rows (same formula : pull the raw level towards the pool average
# by as much as it could be luck, luck = ball by ball luck times the factor for balls of one match moving together)
def shrunk_level(rows, table, in_pool, weight=None):
    weight = np.ones(len(rows)) if weight is None else weight
    code = rows['code'].values
    balls, raw = table['balls'].values, table['raw_level'].values
    in_pool = in_pool & (balls > 0)
    ball_spread = (table['squares'].sum() - (table['total'] ** 2 / table['balls']).sum()) / (balls.sum() - 1)
    luck = ball_spread / np.where(balls > 0, balls, np.nan)
    in_match = (rows['total'].values - rows['balls'].values * raw[code]) / balls[code]
    by_match = np.bincount(code, weight * in_match ** 2, minlength=len(table))
    luck = luck * by_match[in_pool].mean() / luck[in_pool].mean()
    spread = max(raw[in_pool].var(ddof=1) - luck[in_pool].mean(), 0)
    centre = raw[in_pool].mean()
    return centre + (raw - centre) * spread / (spread + luck)


# Who is ranked : the step 4 rules. min_high high pressure balls and min_matches pressure matches,
# and half of both in the IPL itself. The pool for shrinking is everyone with min_high high pressure balls.
def who_is_ranked(rows, players, min_high, min_matches=10):
    everything = player_numbers(rows, players)
    in_ipl = player_numbers(rows[rows['comp_group'] == 'IPL'], players)
    in_pool = (everything['high_balls'] >= min_high).values
    eligible = in_pool & (everything['pressure_matches'] >= min_matches).values \
        & (in_ipl['high_balls'] >= min_high / 2).values & (in_ipl['pressure_matches'] >= min_matches / 2).values
    return in_pool, eligible


# Batters on legal balls, bowlers on every delivery (as in step 4)
groups = {}
for label, source, player, value, min_high in [
    ('batters', balls[balls['legal']], 'striker_id', 'batter_value', 100),
    ('bowlers', balls, 'bowler_id', 'bowler_value', 120),
]:
    rows, ids = match_rows(source, player, value)
    in_pool, eligible = who_is_ranked(rows, len(ids), min_high)
    table = player_numbers(rows, len(ids))
    table['level'] = shrunk_level(rows, table, in_pool)
    groups[label] = {'rows': rows, 'ids': ids, 'min_high': min_high, 'in_pool': in_pool, 'eligible': eligible, 'table': table}

    # Check : the numbers built from the match rows must be the step 4 numbers
    mine = table[eligible].set_index(ids[eligible])
    old = step4[label]
    print(f"{label}: {len(rows)} match rows, {len(ids)} players, {in_pool.sum()} in the pool, {eligible.sum()} ranked. "
          f"Same players as step 4: {set(mine.index) == set(old.index)}, "
          f"biggest gap in level {(mine['level'] - old['level']).abs().max():.5f}, in consistency {(mine['beat_average'] - old['beat_average']).abs().max():.5f}, "
          f"same top 10: {list(mine['level'].sort_values(ascending=False).index[:10]) == list(old.index[:10])}")

batters: 13229 match rows, 824 players, 121 in the pool, 57 ranked. Same players as step 4: True, biggest gap in level 0.00005, in consistency 0.00005, same top 10: True
bowlers: 9730 match rows, 596 players, 102 in the pool, 62 ranked. Same players as step 4: True, biggest gap in level 0.00005, in consistency 0.00005, same top 10: True


In [30]:
# Test 1 : split-half
# I split the matches into two random halves, 100 times. In every split each ranked player is measured in
# both halves, and I take the correlation between the halves over the ranked players.
#   level               : value per ball over all his balls
#   high pressure value : value per ball on his high pressure balls only
#   clutch              : high pressure balls minus his other balls. If it is a skill, it must come back in the other half
#   consistency         : share of his pressure matches above the average player
# Then the question that decides the ranking : what predicts the high pressure value of the OTHER half best?
# score = w * level + (1 - w) * high pressure value, for w from 0 (only high pressure balls) to 1 (only level)
splitter = np.random.RandomState(5)
blend_weights = np.round(np.arange(0, 1.01, 0.1), 1)
same_table, blend_table = {}, {}
for label, group in groups.items():
    rows, players, eligible = group['rows'], len(group['ids']), group['eligible']
    rule_rows = rows.assign(high_balls=rows['rule_balls'], high_total=rows['rule_total'])
    match_code = rows['match_code'].values
    same, blend = [], []
    for run in range(100):
        first = (splitter.rand(match_code.max() + 1) < 0.5)[match_code]
        halves = [player_numbers(rows[side], players)[eligible] for side in [first, ~first]]
        rule_halves = [player_numbers(rule_rows[side], players)[eligible] for side in [first, ~first]]
        same.append({
            'level': halves[0]['raw_level'].corr(halves[1]['raw_level']),
            'high pressure value': halves[0]['raw_high'].corr(halves[1]['raw_high']),
            'clutch': halves[0]['delta'].corr(halves[1]['delta']),
            'clutch (rule index)': rule_halves[0]['delta'].corr(rule_halves[1]['delta']),
            'consistency': halves[0]['beat_average'].corr(halves[1]['beat_average']),
        })
        for one, other in [(0, 1), (1, 0)]:
            blend.append([(w * halves[one]['raw_level'] + (1 - w) * halves[one]['raw_high']).corr(halves[other]['raw_high'])
                          for w in blend_weights])
    same, blend = pd.DataFrame(same), np.array(blend)
    same_table[label + ' average'] = same.mean()
    same_table[label + ' spread'] = same.std()
    same_table[label + ' above 0'] = (same > 0).mean()
    blend_table[label + ' average'] = blend.mean(axis=0)
    blend_table[label + ' spread'] = blend.std(axis=0)
    blend_table[label + ' best in'] = np.bincount(blend.argmax(axis=1), minlength=len(blend_weights))

print("Correlation between the two halves (100 splits : average, spread from split to split, share of splits above 0)")
print(pd.DataFrame(same_table).round(3).to_string())
print()
print("Correlation of the blend with the high pressure value of the other half (200 = 100 splits, both directions)")
print(pd.DataFrame(blend_table, index=pd.Index(blend_weights, name='weight on level')).round(3).to_string())

Correlation between the two halves (100 splits : average, spread from split to split, share of splits above 0)
                     batters average  batters spread  batters above 0  bowlers average  bowlers spread  bowlers above 0
level                          0.362           0.088             1.00            0.198           0.098             0.98
high pressure value           -0.040           0.100             0.34            0.212           0.095             0.99
clutch                        -0.243           0.093             0.01            0.073           0.093             0.78
clutch (rule index)           -0.043           0.113             0.38            0.007           0.094             0.57
consistency                    0.083           0.086             0.80            0.271           0.088             1.00

Correlation of the blend with the high pressure value of the other half (200 = 100 splits, both directions)
                 batters average  batters spread  batters be

In [31]:
# Saving the two tables above for the charts of step 6
pd.DataFrame(same_table).round(4).to_csv(os.path.join(output_folder, "05_splithalf.csv"))
pd.DataFrame(blend_table, index=pd.Index(blend_weights, name='weight on level')).round(4).to_csv(os.path.join(output_folder, "05_blend.csv"))

In [32]:
# How far from 0 can the clutch correlation be by chance?
# Same trick as step 4 : inside every player I shuffle which of his balls are called high pressure, so nobody
# can have a clutch skill. Then the same split-half (20 splits per shuffle, 30 shuffles).
shuffler = np.random.RandomState(7)
for label, source, player, value in [
    ('batters', balls[balls['legal']], 'striker_id', 'batter_value'),
    ('bowlers', balls, 'bowler_id', 'bowler_value'),
]:
    players, eligible = len(groups[label]['ids']), groups[label]['eligible']
    by_chance = []
    for run in range(30):
        test = source.assign(high=source.groupby(player)['high'].transform(lambda x: shuffler.permutation(x.values)))
        rows, ids = match_rows(test, player, value)
        match_code = rows['match_code'].values
        found = []
        for split in range(20):
            first = (shuffler.rand(match_code.max() + 1) < 0.5)[match_code]
            halves = [player_numbers(rows[side], players)[eligible] for side in [first, ~first]]
            found.append(halves[0]['delta'].corr(halves[1]['delta']))
        by_chance.append(np.mean(found))
    real = same_table[label + ' average']['clutch']
    print(f"{label}: clutch correlation with shuffled labels, average {np.mean(by_chance):+.3f}, from {min(by_chance):+.3f} to {max(by_chance):+.3f}. "
          f"Real: {real:+.3f}, {np.mean(np.array(by_chance) <= real):.0%} of the shuffles are as low, {np.mean(np.array(by_chance) >= real):.0%} as high")

batters: clutch correlation with shuffled labels, average -0.003, from -0.256 to +0.228. Real: -0.243, 3% of the shuffles are as low, 97% as high
bowlers: clutch correlation with shuffled labels, average +0.011, from -0.235 to +0.248. Real: +0.073, 70% of the shuffles are as low, 30% as high


In [33]:
# Test 2 : IPL against T20I
# The level again, from the balls of one competition only. Three questions :
# 1. Do the two competitions agree? Correlation of a player's IPL level with his T20I level (players with 300+ balls in both)
# 2. Is the list carried by T20I balls? The top 10 from IPL balls only against my top 10
# 3. Is a T20I ball easier? gap = how much better the same player is per ball in T20Is than in the IPL
#    (pool players with 300+ balls in both, a player counts for more if both his numbers have many balls behind them).
#    I take the gap off every T20I ball and rank again : the list in IPL terms. The final lists use this.
for label, group in groups.items():
    rows, ids, in_pool, eligible = group['rows'], group['ids'], group['in_pool'], group['eligible']
    one_comp = {}
    for comp in ['IPL', 'T20I']:
        part = rows[rows['comp_group'] == comp]
        table = player_numbers(part, len(ids))
        table['level'] = shrunk_level(part, table, in_pool & (table['balls'] >= 300).values)
        one_comp[comp] = table.set_index(ids)
    ipl, t20i = one_comp['IPL'], one_comp['T20I']
    both = (ipl['balls'] >= 300) & (t20i['balls'] >= 300)
    ranked_both = both & eligible
    print(f"{label}: {ranked_both.sum()} of the {eligible.sum()} ranked players have 300+ balls in both. "
          f"Correlation of IPL level with T20I level {ipl.loc[ranked_both, 'raw_level'].corr(t20i.loc[ranked_both, 'raw_level']):.2f} "
          f"(rank correlation {ipl.loc[ranked_both, 'raw_level'].corr(t20i.loc[ranked_both, 'raw_level'], method='spearman'):.2f})")

    mine = list(step4[label].index[:10])
    ipl_only = list(ipl.loc[eligible, 'level'].sort_values(ascending=False).index[:10])
    print(f"   top 10 from IPL balls only: {len(set(mine) & set(ipl_only))} of my 10 stay. "
          f"Out: {names[label][[p for p in mine if p not in ipl_only]].tolist()}. In: {names[label][[p for p in ipl_only if p not in mine]].tolist()}")
    print(f"   rank correlation of the IPL only level with my score, all ranked players: {ipl.loc[eligible, 'level'].corr(step4[label]['score'], method='spearman'):.2f}")
    share = (ipl['balls'] / (ipl['balls'] + t20i['balls'])).loc[mine]
    print(f"   share of IPL balls in my top 10: {dict(zip(names[label][mine], share.round(2)))}")

    pool_both = both & in_pool
    gaps = (t20i['raw_level'] - ipl['raw_level'])[pool_both]
    gap = np.average(gaps, weights=(1 / (1 / ipl['balls'] + 1 / t20i['balls']))[pool_both])
    is_t20i = rows['comp_group'] == 'T20I'
    ipl_terms = rows.assign(total=rows['total'] - gap * rows['balls'] * is_t20i,
                            squares=rows['squares'] + (gap ** 2 * rows['balls'] - 2 * gap * rows['total']) * is_t20i)
    group['ipl_rows'] = ipl_terms
    level = pd.Series(shrunk_level(ipl_terms, player_numbers(ipl_terms, len(ids)), in_pool), index=ids)[eligible].sort_values(ascending=False)
    top = list(level.index[:10])
    print(f"   the same player is {gap:+.3f} per ball better in T20Is ({pool_both.sum()} players, plain average {gaps.mean():+.3f}, give or take {gaps.std() / np.sqrt(len(gaps)):.3f}). "
          f"With that taken off every T20I ball: {len(set(mine) & set(top))} of my 10 stay. "
          f"Out: {names[label][[p for p in mine if p not in top]].tolist()}. In: {names[label][[p for p in top if p not in mine]].tolist()}")
    print(f"   top 10 in IPL terms: {names[label][top].tolist()}")
    print(f"   next 3: {names[label][list(level.index[10:13])].tolist()}, level of place 10, 11: {level.iloc[9]:.3f}, {level.iloc[10]:.3f}")

batters: 37 of the 57 ranked players have 300+ balls in both. Correlation of IPL level with T20I level 0.32 (rank correlation 0.27)
   top 10 from IPL balls only: 8 of my 10 stay. Out: ['Jos Buttler', 'Mitchell Marsh']. In: ['Shreyas Iyer', 'Shubman Gill']
   rank correlation of the IPL only level with my score, all ranked players: 0.90
   share of IPL balls in my top 10: {'Abhishek Sharma': 0.54, 'Suryakumar Yadav': 0.49, 'Tim David': 0.53, 'Heinrich Klaasen': 0.85, 'Phil Salt': 0.44, 'Rajat Patidar': 1.0, 'Travis Head': 0.68, 'Sai Sudharsan': 1.0, 'Jos Buttler': 0.63, 'Mitchell Marsh': 0.61}
   the same player is +0.099 per ball better in T20Is (40 players, plain average +0.094, give or take 0.028). With that taken off every T20I ball: 10 of my 10 stay. Out: []. In: []
   top 10 in IPL terms: ['Abhishek Sharma', 'Heinrich Klaasen', 'Suryakumar Yadav', 'Tim David', 'Rajat Patidar', 'Phil Salt', 'Sai Sudharsan', 'Travis Head', 'Jos Buttler', 'Mitchell Marsh']
   next 3: ['David Warner'

In [34]:
# Test 3 : the rule index instead of my pressure score
# The lists are ranked on level, and the level does not use the pressure score at all. Pressure only decides
# who has enough high pressure balls and pressure matches to be ranked. So I pick the top fifth of balls with
# the simple rule index of step 2 (only 7% of all balls are high in both), and build the lists again.
for label, group in groups.items():
    rows, ids, eligible = group['rows'], group['ids'], group['eligible']
    rule_rows = rows.assign(high_balls=rows['rule_balls'], high_total=rows['rule_total'])
    rule_pool, rule_eligible = who_is_ranked(rule_rows, len(ids), group['min_high'])
    table = player_numbers(rule_rows, len(ids))
    table['level'] = shrunk_level(rule_rows, table, rule_pool)
    rule_top = list(table[rule_eligible].set_index(ids[rule_eligible])['level'].sort_values(ascending=False).index[:10])
    mine = list(step4[label].index[:10])
    out = [p for p in mine if p not in rule_top]
    print(f"{label}: {rule_eligible.sum()} players ranked with the rule index ({(eligible & rule_eligible).sum()} of my {eligible.sum()}). "
          f"{len(set(mine) & set(rule_top))} of my top 10 stay. "
          f"Out: {names[label][out].tolist()}. In: {names[label][[p for p in rule_top if p not in mine]].tolist()}")
    print(f"   of those out, not ranked at all with the rule index (too few high balls or pressure matches): "
          f"{names[label][[p for p in out if p not in ids[rule_eligible]]].tolist()}")

    # The value on high pressure balls itself, under the two definitions
    in_both = eligible & rule_eligible
    print(f"   rank correlation of the raw high pressure value under the two definitions: "
          f"{group['table'].loc[in_both, 'raw_high'].corr(table.loc[in_both, 'raw_high'], method='spearman'):.2f}")

batters: 44 players ranked with the rule index (40 of my 57). 7 of my top 10 stay. Out: ['Abhishek Sharma', 'Rajat Patidar', 'Travis Head']. In: ['Andre Russell', 'Namboori Tilak Varma', 'Shreyas Iyer']
   of those out, not ranked at all with the rule index (too few high balls or pressure matches): ['Abhishek Sharma', 'Rajat Patidar', 'Travis Head']
   rank correlation of the raw high pressure value under the two definitions: 0.45
bowlers: 63 players ranked with the rule index (56 of my 62). 10 of my top 10 stay. Out: []. In: []
   of those out, not ranked at all with the rule index (too few high balls or pressure matches): []
   rank correlation of the raw high pressure value under the two definitions: 0.49


In [35]:
# Test 4 : other eligibility numbers
# My pick was 100 (batters) or 120 (bowlers) high pressure balls and 10 pressure matches, half of both in the IPL.
# The same lists with lower and higher minimums : how many players are ranked, and how many of my top 10 stay?
for label, group in groups.items():
    rows, ids = group['rows'], group['ids']
    mine = list(step4[label].index[:10])
    for min_high in [group['min_high'] * 0.6, group['min_high'], group['min_high'] * 1.5]:
        for min_matches in [5, 10, 15]:
            in_pool, eligible = who_is_ranked(rows, len(ids), min_high, min_matches)
            level = pd.Series(shrunk_level(rows, group['table'], in_pool), index=ids)[eligible]
            top = list(level.sort_values(ascending=False).index[:10])
            print(f"{label}: {int(min_high)}+ high pressure balls, {min_matches}+ pressure matches -> {eligible.sum()} ranked, "
                  f"{len(set(mine) & set(top))} of my top 10 stay. "
                  f"Out: {names[label][[p for p in mine if p not in top]].tolist()}. In: {names[label][[p for p in top if p not in mine]].tolist()}")

batters: 60+ high pressure balls, 5+ pressure matches -> 91 ranked, 8 of my top 10 stay. Out: ['Jos Buttler', 'Mitchell Marsh']. In: ['Vaibhav Sooryavanshi', 'Andre Russell']
batters: 60+ high pressure balls, 10+ pressure matches -> 57 ranked, 10 of my top 10 stay. Out: []. In: []
batters: 60+ high pressure balls, 15+ pressure matches -> 27 ranked, 7 of my top 10 stay. Out: ['Heinrich Klaasen', 'Phil Salt', 'Rajat Patidar']. In: ['Namboori Tilak Varma', 'Shreyas Iyer', 'Nicholas Pooran']
batters: 100+ high pressure balls, 5+ pressure matches -> 77 ranked, 8 of my top 10 stay. Out: ['Jos Buttler', 'Mitchell Marsh']. In: ['Vaibhav Sooryavanshi', 'Andre Russell']
batters: 100+ high pressure balls, 10+ pressure matches -> 57 ranked, 10 of my top 10 stay. Out: []. In: []
batters: 100+ high pressure balls, 15+ pressure matches -> 27 ranked, 7 of my top 10 stay. Out: ['Heinrich Klaasen', 'Phil Salt', 'Rajat Patidar']. In: ['Namboori Tilak Varma', 'Cameron Green', 'Shreyas Iyer']
batters: 150+

In [36]:
# Test 5 : two choices of mine that move the numbers, and the shrinking
# 1. The price of a wicket. A wicket is half of the variance of a ball value (step 3), and the price (runs a wicket costs) is
#    my own estimate. I rank again with the price times 0, 0.5, 1.5 and 2. Times 0 = runs only.
# 2. The years. The value is an average over 2022 to 2026 with every year counted the same. I rank again on 2024 to 2026
#    and on 2025 to 2026 only (the pool for shrinking and the minimums stay as they are).
# 3. No shrinking : the raw level of every ranked player.
extra = pd.read_parquet(os.path.join(output_folder, "balls_clean.parquet"), columns=keys + ['bat_runs', 'bowler_runs', 'striker_out', 'bowler_wicket', 'match_date'])
extra = extra.merge(pd.read_parquet(os.path.join(output_folder, "balls_value.parquet"), columns=keys + [
    'wicket_cost', 'exp_bat_runs', 'exp_bat_out', 'exp_bowl_runs', 'exp_bowl_wicket']), on=keys)
more = balls.merge(extra, on=keys)
more['year'] = pd.to_datetime(more['match_date']).dt.year

sensitivity = []
for label, source, player in [('batters', more[more['legal']], 'striker_id'), ('bowlers', more, 'bowler_id')]:
    group = groups[label]
    ids, in_pool, eligible = group['ids'], group['in_pool'], group['eligible']
    mine = list(step4[label].index[:10])
    base = pd.Series(shrunk_level(group['rows'], group['table'], in_pool), index=ids)[eligible]
    scenarios = [('wicket price x 0 (runs only)', 0.0, 2022, True), ('wicket price x 0.5', 0.5, 2022, True), ('wicket price x 1.5', 1.5, 2022, True),
                 ('wicket price x 2', 2.0, 2022, True), ('years 2024 to 2026 only', 1.0, 2024, True), ('years 2025 to 2026 only', 1.0, 2025, True),
                 ('no shrinking (raw level)', 1.0, 2022, False)]
    for name, price, first_year, shrink in scenarios:
        if label == 'batters':
            value = source['bat_runs'] - source['exp_bat_runs'] - price * source['wicket_cost'] * (source['striker_out'].astype(float) - source['exp_bat_out'])
        else:
            value = source['exp_bowl_runs'] - source['bowler_runs'] + price * source['wicket_cost'] * (source['bowler_wicket'].astype(float) - source['exp_bowl_wicket'])
        part = source.assign(changed=value)[source['year'] >= first_year]
        rows, _ = match_rows(part, player, 'changed')
        rows['code'] = np.searchsorted(ids, rows[player])
        table = player_numbers(rows, len(ids))
        has_balls = (table['balls'] > 0).values
        level = shrunk_level(rows, table, in_pool & has_balls) if shrink else table['raw_level'].values
        level = pd.Series(level, index=ids)[eligible & has_balls].sort_values(ascending=False)
        top = list(level.index[:10])
        sensitivity.append({'list': label, 'scenario': name, 'ranked': len(level), 'of_my_10_stay': len(set(top) & set(mine)),
                            'rank_correlation': level.corr(base.reindex(level.index), method='spearman'),
                            'out': ", ".join(names[label][[p for p in mine if p not in top]]),
                            'in': ", ".join(names[label][[p for p in top if p not in mine]])})
sensitivity = pd.DataFrame(sensitivity)
sensitivity.round(3).to_csv(os.path.join(output_folder, "05_sensitivity.csv"), index=False)
pd.set_option('display.width', 250)
pd.set_option('display.max_colwidth', 90)
print(sensitivity.round(2).to_string(index=False))

# How often does a player of my final 10 drop out of the top 10 in these 7 runs?
for label in ['batters', 'bowlers']:
    part = sensitivity[sensitivity['list'] == label]
    dropped = pd.Series(", ".join(part['out']).split(", ")).replace('', np.nan).dropna().value_counts()
    print(f"{label}: times each player drops out in the 7 runs: {dropped.to_dict()}")

   list                     scenario  ranked  of_my_10_stay  rank_correlation                                             out                                                     in
batters wicket price x 0 (runs only)      57              7              0.76      Sai Sudharsan, Jos Buttler, Mitchell Marsh     Prabhsimran Singh, Ryan Rickelton, Nicholas Pooran
batters           wicket price x 0.5      57              8              0.94                      Sai Sudharsan, Jos Buttler                         Cameron Green, Nicholas Pooran
batters           wicket price x 1.5      57              8              0.96                     Travis Head, Mitchell Marsh                     Namboori Tilak Varma, Shubman Gill
batters             wicket price x 2      57              7              0.89          Phil Salt, Travis Head, Mitchell Marsh        Virat Kohli, Lokesh Rahul, Namboori Tilak Varma
batters      years 2024 to 2026 only      57              7              0.84    Suryakumar Yad

In [37]:
# Final lists : ranked on the level in IPL terms (test 2 : the T20I gap taken off every T20I ball),
# because the lists are for an IPL auction. The players are the same 20 as in step 4, only the order moves.
#
# Rank confidence : the matches I have are one draw of the matches that could have been played. I draw the same
# number of matches again with repeats (some matches twice, some not at all), 1,000 times, and rank the players
# again every time. The ranked players, the pool and the T20I gap stay the same, so this shows only how far luck
# in the matches moves a rank.
#   top10_share : in how many of the 1,000 draws he is in the top 10
#   place_low / place_high : 90% of the draws put him between these two places
drawer = np.random.RandomState(11)
final = {}
for label, group in groups.items():
    rows, ids, in_pool, eligible = group['ipl_rows'], group['ids'], group['in_pool'], group['eligible']
    ipl_level = shrunk_level(rows, player_numbers(rows, len(ids)), in_pool)[eligible]
    match_code = rows['match_code'].values
    matches = match_code.max() + 1
    places = np.zeros((1000, eligible.sum()), dtype=int)
    for run in range(1000):
        weight = drawer.multinomial(matches, np.ones(matches) / matches)[match_code].astype(float)
        level = shrunk_level(rows, player_numbers(rows, len(ids), weight), in_pool, weight)[eligible]
        places[run] = (-level).argsort().argsort() + 1

    table = step4[label].rename(columns={'rank': 'step4_rank'}).join(pd.DataFrame({
        'ipl_level': ipl_level,
        'top10_share': (places <= 10).mean(axis=0),
        'place_median': np.median(places, axis=0),
        'place_low': np.percentile(places, 5, axis=0),
        'place_high': np.percentile(places, 95, axis=0),
    }, index=ids[eligible]))
    table = table.sort_values('ipl_level', ascending=False)
    table.insert(0, 'rank', np.arange(1, len(table) + 1))
    table.round(4).to_csv(os.path.join(output_folder, f"05_{label}.csv"))
    final[label] = table

    top, rest = table.head(10), table.iloc[10:]
    print(f"{label}: same 10 names as step 4: {set(top.index) == set(step4[label].index[:10])}. "
          f"{int((top['top10_share'] >= 0.5).sum())} of the top 10 are in the top 10 in at least half of the draws. "
          f"Outside the top 10 with a share of 25% or more: {rest[rest['top10_share'] >= 0.25]['name'].tolist()}")
    print(table[['rank', 'step4_rank', 'name', 'balls', 'level', 'ipl_level', 'top10_share', 'place_median', 'place_low', 'place_high']].head(15).round(3).to_string(index=False))
    print()

batters: same 10 names as step 4: True. 8 of the top 10 are in the top 10 in at least half of the draws. Outside the top 10 with a share of 25% or more: []
 rank  step4_rank             name  balls  level  ipl_level  top10_share  place_median  place_low  place_high
    1           1  Abhishek Sharma   2239  0.307      0.271        0.971           1.0        1.0         8.0
    2           4 Heinrich Klaasen   1408  0.229      0.213        0.976           3.0        1.0         8.0
    3           2 Suryakumar Yadav   2746  0.248      0.204        0.942           4.0        1.0        11.0
    4           3        Tim David   1210  0.232      0.195        0.889           4.0        1.0        15.0
    5           6    Rajat Patidar    916  0.187      0.176        0.871           5.0        1.0        16.0
    6           5        Phil Salt   1628  0.199      0.153        0.675           7.0        2.0        28.0
    7           8    Sai Sudharsan   1674  0.154      0.145        0.704  

### What step 5 found

- Level comes back in the other half, clutch does not. Level: 0.36 for batters and 0.20 for bowlers, above 0 in almost every split. Clutch: -0.24 for batters and +0.07 for bowlers. With shuffled labels (no clutch possible) the same number runs from about -0.25 to +0.25, so neither is a skill I can see. The batter number is at the low end of chance: a batter who looked clutch in one half looked the opposite in the other.
- Level is the number to rank on. For batters it predicts the high pressure value of the other half at 0.21, against -0.04 for the high pressure balls themselves, and it is the best weight in 196 of 200 tries. For bowlers level (0.22), high pressure balls only (0.21) and the best blend (0.24) are all inside the spread of 0.10, so there is no reason to move.
- T20I balls are easier. The same player is 0.10 (batters) and 0.04 (bowlers) runs per ball better in T20Is. The final lists take that off every T20I ball, because they are for an IPL auction. The 20 names stay the same as step 4, only the order moves by one or two places. On IPL balls alone 8 batters and 7 bowlers stay.
- With the rule index, all 10 bowlers stay. 7 batters stay, and the 3 who go (Abhishek Sharma, Rajat Patidar, Travis Head) are not ranked at all with the rule index because they have fewer than 10 pressure matches under it.
- The ball minimum hardly matters. The match minimum does for batters: with 5 matches Vaibhav Sooryavanshi and Andre Russell come in, with 15 Klaasen, Salt and Patidar are no longer ranked. The bowler list keeps 9 or 10 names everywhere.
- My own choices move the lists by 1 to 3 names, never more. With the price of a wicket times 0, 0.5, 1.5 or 2, or with only the last two or three seasons, 7 to 9 of each top 10 stay. Abhishek Sharma, Klaasen, Tim David and Patidar (batters) and Bumrah and Ellis (bowlers) never drop out. Head drops out in 4 of 7 runs and Rashid Khan in 4 of 7. Kohli (17th) enters the top 10 when a wicket costs twice as much, or on 2024 to 2026 only. Shrinking changes almost nothing between players with similar ball counts.
- The top of both final lists is firm and the last places are close. Batters 1 to 7 are in the top 10 in 68% to 98% of the draws, places 8 to 10 in 32% to 52%. Bowlers 1 to 5 in 71% to 94%, places 6 to 10 in 30% to 60%. Lockie Ferguson (37%) and Kagiso Rabada (31%) are the closest players outside.